# STOP Ablation Experiments

Tests which feature representation best separates STOP vs CONTINUE segments.
Loads all data from caches populated by notebook 02.

In [1]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

if not os.path.isdir('src'):
    os.chdir(os.path.join(os.getcwd(), '..', '..'))

import sys
sys.path.insert(0, 'src')

import json
import numpy as np
import torch
from pathlib import Path

MC_DIR = Path('data/masterclass_pipeline')
COMPOSITE_DIR = Path('data/composite_labels')
CACHE_DIR = Path('data/masterclass_cache')

print(f'Working directory: {os.getcwd()}')

Working directory: /Users/jdhiman/Documents/crescendai/model


## Load segments, embeddings, and scores from cache

In [ ]:
from audio_experiments.constants import PERCEPIANO_DIMENSIONS
from masterclass_experiments.data import load_moments, identify_segments
from masterclass_experiments.features import stats_pool
from masterclass_experiments.bridge import compute_weights, compute_composite_labels
from masterclass_experiments.scoring import PERCEPIANO_MUQ_R2
from masterclass_experiments.evaluation import leave_one_video_out_cv

dim_index = {d: i for i, d in enumerate(PERCEPIANO_DIMENSIONS)}

DIM_MAPPING = {
    'dynamics':       ['dynamic_range', 'timbre_loudness'],
    'timing':         ['timing', 'tempo'],
    'pedaling':       ['pedal_amount', 'pedal_clarity'],
    'articulation':   ['articulation_length', 'articulation_touch'],
    'phrasing':       ['space', 'mood_imagination'],
    'interpretation': ['balance', 'timbre_variety', 'timbre_depth',
                       'mood_valence', 'interpretation', 'sophistication'],
}
dim_names = list(DIM_MAPPING.keys())
weights = compute_weights(DIM_MAPPING, PERCEPIANO_MUQ_R2)

# Reconstruct segments
moments = load_moments(MC_DIR / 'all_moments.jsonl')
segments = identify_segments(moments)
n_stop = sum(1 for s in segments if s.label == 'stop')
print(f'{len(segments)} segments ({n_stop} stop, {len(segments) - n_stop} continue)')

# Load cached quality scores (19-dim)
scores_dir = CACHE_DIR / 'quality_scores'
quality_scores = {}
for seg in segments:
    pt_path = scores_dir / f'{seg.segment_id}.pt'
    if not pt_path.exists():
        raise FileNotFoundError(f'Missing quality score: {pt_path}. Run notebook 02 first.')
    quality_scores[seg.segment_id] = torch.load(pt_path, weights_only=True)
print(f'Loaded {len(quality_scores)} quality scores from cache')

# Load cached raw MuQ embeddings ([T, 1024])
emb_dir = CACHE_DIR / 'muq_embeddings'
raw_embeddings = {}
for seg in segments:
    pt_path = emb_dir / f'{seg.segment_id}.pt'
    if not pt_path.exists():
        raise FileNotFoundError(f'Missing embedding: {pt_path}. Run notebook 02 first.')
    raw_embeddings[seg.segment_id] = torch.load(pt_path, weights_only=True)
print(f'Loaded {len(raw_embeddings)} raw MuQ embeddings from cache')

# Compute composites
mc_pp_labels = {sid: score.numpy() for sid, score in quality_scores.items()}
mc_composites = compute_composite_labels(mc_pp_labels, weights, dim_index)

## Ablation: feature sets and class weighting

In [3]:
# Shared labels and video IDs
valid_segments = [s for s in segments if s.segment_id in quality_scores]
y = np.array([1 if s.label == 'stop' else 0 for s in valid_segments])
vid = np.array([s.video_id for s in valid_segments])
sid = np.array([s.segment_id for s in valid_segments])

ablations = {}

# Baseline: 6-dim composites (unweighted)
X_comp = np.array([[mc_composites[s.segment_id][d] for d in dim_names] for s in valid_segments])
ablations['6-dim composite'] = leave_one_video_out_cv(X_comp, y, vid, sid)

# (A) Raw 19-dim PercePiano scores
X_pp19 = np.array([quality_scores[s.segment_id].numpy() for s in valid_segments])
ablations['19-dim PercePiano'] = leave_one_video_out_cv(X_pp19, y, vid, sid)

# (B) Pooled MuQ [2048] features
X_muq = np.array([stats_pool(raw_embeddings[s.segment_id]).numpy() for s in valid_segments])
ablations['2048-dim MuQ pooled'] = leave_one_video_out_cv(X_muq, y, vid, sid)

# (C) 6-dim composites with class_weight='balanced'
ablations['6-dim composite (balanced)'] = leave_one_video_out_cv(
    X_comp, y, vid, sid, class_weight='balanced',
)

# Summary table
print(f"{'Experiment':<30s} {'AUC':>6s} {'Acc':>6s} {'Prec':>6s} {'Rec':>6s}")
print('-' * 60)
for name, res in ablations.items():
    print(f"{name:<30s} {res['auc']:6.3f} {res['accuracy']:6.3f} {res['precision']:6.3f} {res['recall']:6.3f}")

best_name = max(ablations, key=lambda k: ablations[k]['auc'])
print(f"\nBest: {best_name} (AUC {ablations[best_name]['auc']:.3f})")

Experiment                        AUC    Acc   Prec    Rec
------------------------------------------------------------
6-dim composite                 0.644  0.832  0.833  0.999
19-dim PercePiano               0.757  0.841  0.846  0.989
2048-dim MuQ pooled             0.845  0.848  0.909  0.909
6-dim composite (balanced)      0.649  0.611  0.893  0.605

Best: 2048-dim MuQ pooled (AUC 0.845)


## Per-video AUC analysis

In [4]:
from sklearn.metrics import roc_auc_score

# Use best ablation's per-segment predictions
best_res = ablations[best_name]
preds = best_res['per_segment']

# Group by video
video_preds = {}
for p in preds:
    vid_id = p['video_id']
    video_preds.setdefault(vid_id, {'y_true': [], 'y_proba': []})
    video_preds[vid_id]['y_true'].append(p['y_true'])
    video_preds[vid_id]['y_proba'].append(p['y_pred_proba'])

# Compute per-video AUC (only possible when video has both classes)
per_video_auc = {}
single_class_videos = []
for vid_id, data in sorted(video_preds.items()):
    yt = np.array(data['y_true'])
    if len(np.unique(yt)) < 2:
        single_class_videos.append((vid_id, int(yt[0]), len(yt)))
        continue
    per_video_auc[vid_id] = roc_auc_score(yt, data['y_proba'])

print(f'Videos with both classes: {len(per_video_auc)} / {len(video_preds)}')
print(f'Videos with single class only: {len(single_class_videos)}')

if per_video_auc:
    aucs = np.array(list(per_video_auc.values()))
    print(f'\nPer-video AUC stats: mean={aucs.mean():.3f}, std={aucs.std():.3f}, '
          f'min={aucs.min():.3f}, max={aucs.max():.3f}')

    # Top/bottom 5
    sorted_vids = sorted(per_video_auc.items(), key=lambda kv: kv[1], reverse=True)
    print(f'\nTop 5 videos by AUC:')
    for vid_id, auc in sorted_vids[:5]:
        n = len(video_preds[vid_id]['y_true'])
        print(f'  {vid_id}: AUC={auc:.3f} ({n} segments)')
    print(f'\nBottom 5 videos by AUC:')
    for vid_id, auc in sorted_vids[-5:]:
        n = len(video_preds[vid_id]['y_true'])
        print(f'  {vid_id}: AUC={auc:.3f} ({n} segments)')

Videos with both classes: 60 / 60
Videos with single class only: 0

Per-video AUC stats: mean=0.832, std=0.162, min=0.214, max=1.000

Top 5 videos by AUC:
  ALDzxU452gA: AUC=1.000 (20 segments)
  HQ_NYQT-ZT4: AUC=1.000 (19 segments)
  fR51x3LMbhs: AUC=1.000 (23 segments)
  nt5jf0X90J4: AUC=1.000 (13 segments)
  rSISZncZ6QI: AUC=0.994 (30 segments)

Bottom 5 videos by AUC:
  ox7wlxfilYw: AUC=0.562 (18 segments)
  levyvp6QI1U: AUC=0.500 (26 segments)
  I3a2IsqDjRw: AUC=0.468 (33 segments)
  qhDFutPI7eE: AUC=0.377 (26 segments)
  1tQlwsm8xeA: AUC=0.214 (17 segments)
